In [0]:
# Databricks Notebook: ACID Proof without DBFS Root Access
from pyspark.sql.functions import col, lit, when, raise_error

# --- SETUP ---
# We use the catalog/schema instead of raw paths to avoid DBFS Root restrictions
TABLE_NAME = "acid_proof_demo"

print(f"--- PREPARING SECURE PROOF FOR: {TABLE_NAME} ---")
spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")

# Create Version 0
spark.range(10).withColumn("data", lit("Initial")).write.format("delta").saveAsTable(TABLE_NAME)
result_df = spark.sql(f"SELECT * FROM {TABLE_NAME} LIMIT 5")
result_df.show()

# --- 1. ATOMICITY ---
print("\n[A]TOMICITY: Verifying partial writes are impossible.")
initial_count = spark.table(TABLE_NAME).count()
print(f"-> Evidence: Initial Row count is {initial_count}. (Should be 10)")
try:
    # Fail mid-write
    df_crash = spark.range(20).withColumn("data",
        when(col("id") > 15, raise_error("CRASH_NOW")).otherwise(lit("Corrupt")))
    df_crash.write.format("delta").mode("append").saveAsTable(TABLE_NAME)
except Exception:
    print("-> System Crashed Mid-Write (Expected).")

count = spark.table(TABLE_NAME).count()
print(f"-> Evidence: Row count is {count}. (Success if 10)")

# --- 2. CONSISTENCY ---
print("\n[C]ONSISTENCY: Proving Schema Enforcement.")
current_df = spark.sql(f"SELECT * FROM {TABLE_NAME}")
print("-> Evidence: Current Schema.") 
current_df.printSchema()

try:
    bad_df = spark.createDataFrame([(99, "extra", "fail")], ["id", "data", "extra_info"])
    print("-> Evidence: Alter Schema attempt.")
    bad_df.printSchema()
    bad_df.write.format("delta").mode("append").saveAsTable(TABLE_NAME)
except Exception as e:
    print("-> Evidence: Delta Lake blocked invalid schema. (Success)")

consistency_df = spark.sql(f"SELECT * FROM {TABLE_NAME}")
print("-> Evidence: Schema unchanged.")
consistency_df.printSchema()

# --- 3. ISOLATION ---
print("\n[I]SOLATION: Snapshot Isolation.")
init_iso_count = spark.table(TABLE_NAME).count()
print(f"-> Initial Count: {init_iso_count}")
history_df = spark.sql(f"DESCRIBE HISTORY {TABLE_NAME}")
starting_version = history_df.select("version").first()[0]

print(f"-> Snapshot Version: {starting_version}. Deleting data...")
spark.sql(f"DELETE FROM {TABLE_NAME}")

live_count = spark.table(TABLE_NAME).count()
print(f"-> Live Count: {live_count}")

# Read from the version captured BEFORE the delete
isolated_df = spark.read.format("delta").option("versionAsOf", starting_version).table(TABLE_NAME)
print(f"-> Isolated (as of) Reader Count (Version {starting_version}): {isolated_df.count()}")

# --- 4. DURABILITY: TRANSACTION LOG METADATA ---
print("\n[D]URABILITY: Proving the 'Truth' exists in the Metadata.")

# recorded in an immutable JSON log (the Delta Log) on disk
detail_df = spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}")
table_format = detail_df.select("format").first()[0]

print(f"-> Table Format: {table_format}")
print("\n-> Last 3 Transactions in the Immutable Delta Log:")
history_df = spark.sql(f"DESCRIBE HISTORY {TABLE_NAME}")
history_df.select("version", "timestamp", "operation", "operationMetrics", "operationParameters").show(3, truncate=False)

print("-> RESULT: Durability confirmed via Transaction History. Even if the table is 'empty', the log persists.")

--- PREPARING SECURE PROOF FOR: acid_proof_demo ---
+---+-------+
| id|   data|
+---+-------+
|  3|Initial|
|  4|Initial|
|  2|Initial|
|  5|Initial|
|  6|Initial|
+---+-------+


[A]TOMICITY: Verifying partial writes are impossible.
-> Evidence: Initial Row count is 10. (Should be 10)
-> System Crashed Mid-Write (Expected).
-> Evidence: Row count is 10. (Success if 10)

[C]ONSISTENCY: Proving Schema Enforcement.
-> Evidence: Current Schema.
root
 |-- id: long (nullable = true)
 |-- data: string (nullable = true)

-> Evidence: Alter Schema attempt.
root
 |-- id: long (nullable = true)
 |-- data: string (nullable = true)
 |-- extra_info: string (nullable = true)

-> Evidence: Delta Lake blocked invalid schema. (Success)
-> Evidence: Schema unchanged.
root
 |-- id: long (nullable = true)
 |-- data: string (nullable = true)


[I]SOLATION: Snapshot Isolation.
-> Initial Count: 10
-> Snapshot Version: 0. Deleting data...
-> Live Count: 0
-> Isolated (as of) Reader Count (Version 0): 10

[D]